In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb   # これが現在のテーブルコンペの「王道」です
import xgboost as xgb
import catboost as cb

In [66]:
TRAIN_DATA_PATH = os.path.join("data", "train")
TEST_DATA_PATH = os.path.join("data", "test")

def load_data():
    train_df = pd.read_csv(os.path.join(TRAIN_DATA_PATH, "train.csv"))
    train_add=pd.read_csv(os.path.join(TRAIN_DATA_PATH, "train_add.csv"))

    train_stadium=pd.read_csv(os.path.join(TRAIN_DATA_PATH, "stadium.csv"))
    train_stadium = train_stadium.rename(columns={"name": "stadium"})
    train_stadium = train_stadium[["stadium", "capa"]]
    
    

    train_condidion=pd.read_csv(os.path.join(TRAIN_DATA_PATH, "condition.csv"))
    train_condidion_add=pd.read_csv(os.path.join(TRAIN_DATA_PATH, "condition_add.csv"))
    train_condidion = train_condidion[['id', 'weather', 'temperature', 'humidity']]
    train_df = pd.concat([train_df, train_add], axis=0, ignore_index=True)
   
    train_condidion = pd.concat([train_condidion, train_condidion_add], axis=0, ignore_index=True)
    train_df = pd.merge(train_df, train_condidion, on='id', how="left")
    train_df = pd.merge(train_df, train_stadium, on='stadium', how="left")
    test_df = pd.read_csv(os.path.join(TEST_DATA_PATH, "test.csv"))
    test_df = pd.merge(test_df, train_condidion, on='id', how="left")
    test_df = pd.merge(test_df, train_stadium, on='stadium', how="left")
    return train_df, test_df

train_df, test_df = load_data()

In [67]:
train_df.select_dtypes(include=object).head()
train_df['section'] = train_df['match'].apply(lambda x:x.split('節')[0][1:]).astype(int)
test_df['section'] = test_df['match'].apply(lambda x:x.split('節')[0][1:]).astype(int)

train_df['month'] = train_df['gameday'].apply(lambda x:x[:2]).astype(int)
test_df['month'] = test_df['gameday'].apply(lambda x:x[:2]).astype(int)

train_df['weekday'] = train_df['gameday'].apply(lambda x:x[6])
test_df['weekday'] = test_df['gameday'].apply(lambda x:x[6])

train_df['hour'] = train_df['time'].apply(lambda x:x.split(':')[0]).astype(int)
test_df['hour'] = test_df['time'].apply(lambda x:x.split(':')[0]).astype(int)

display(train_df.head(3), test_df.head(3))

/tmp/ipykernel_67970/1849259759.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  train_df.select_dtypes(include=object).head()


,id,y,year,stage,match,gameday,time,home,away,stadium,...,away_07,away_08,away_09,away_10,away_11,capa,section,month,weekday,hour
0,13994,18250,2012,Ｊ１,第１節第１日,03/10(土),14:04,ベガルタ仙台,鹿島アントラーズ,ユアテックスタジアム仙台,...,NaN,NaN,NaN,NaN,NaN,19694,1,3,土,14
1,13995,24316,2012,Ｊ１,第１節第１日,03/10(土),14:04,名古屋グランパス,清水エスパルス,豊田スタジアム,...,NaN,NaN,NaN,NaN,NaN,40000,1,3,土,14
2,13996,17066,2012,Ｊ１,第１節第１日,03/10(土),14:04,ガンバ大阪,ヴィッセル神戸,万博記念競技場,...,NaN,NaN,NaN,NaN,NaN,21000,1,3,土,14


,id,year,stage,match,gameday,time,home,away,stadium,tv,...,away_07,away_08,away_09,away_10,away_11,capa,section,month,weekday,hour
0,15822,2014,Ｊ１,第１８節第１日,08/02(土),19:04,ベガルタ仙台,大宮アルディージャ,ユアテックスタジアム仙台,スカパー！／スカパー！プレミアムサービス,...,NaN,NaN,NaN,NaN,NaN,19694,18,8,土,19
1,15823,2014,Ｊ１,第１８節第１日,08/02(土),18:34,鹿島アントラーズ,サンフレッチェ広島,県立カシマサッカースタジアム,スカパー！／スカパー！プレミアムサービス,...,NaN,NaN,NaN,NaN,NaN,40728,18,8,土,18
2,15824,2014,Ｊ１,第１８節第１日,08/02(土),19:04,浦和レッズ,ヴィッセル神戸,埼玉スタジアム２００２,スカパー！／スカパー！プレミアムサービス／ＮＨＫ ＢＳ１／テレ玉,...,NaN,NaN,NaN,NaN,NaN,63700,18,8,土,19


In [64]:
train_df.drop(train_df[train_df['y']==0].index, inplace=True)

ValueError: could not convert string to float: 'Ｊ１'